# 📅 Time Series Foundations
**fpppy Chapters 1–3 · Pattern Recognition for the Rest of Us**

> A time series is any sequence of observations indexed by time. Before forecasting, you need to understand the structure of your series — trend, seasonality, cycles, and noise. This notebook covers the essentials.

### Required reading
📘 [Forecasting: Principles & Practice (Pythonic) — Ch 1–3](https://otexts.com/fpppy/)

### What you'll learn
- Time series components: trend, seasonality, cycles, remainder
- Time plots, seasonal plots, autocorrelation (ACF/PACF)
- STL decomposition — separating components
- Stationarity — what it means and why it matters
- White noise — the baseline

### Dataset: Air passengers + Australian retail turnover
### Time: ~60 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
import matplotlib.dates as mdates

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: AirPassengers

In [ ]:
import pandas as pd
passengers = pd.read_csv(f'{DATA_DIR}/AirPassengers.csv',
                          parse_dates=['Month'], index_col='Month')
passengers.index.freq = 'MS'
print(f"Air Passengers: {passengers.shape}  ({passengers.index[0].year}–{passengers.index[-1].year})")
passengers.head()

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

## 📊 Part 1 — Time Plots: See the Big Picture

Always start with a time plot. Look for:
- **Trend:** long-run increase or decrease
- **Seasonality:** regular, calendar-related patterns (monthly, quarterly, yearly)
- **Cycles:** irregular longer-term fluctuations (not calendar-based)
- **Outliers/irregularities:** sudden jumps, one-off events

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Raw series
axes[0].plot(passengers.index, passengers['Passengers'], color='#1e5fa8', lw=1.5)
axes[0].fill_between(passengers.index, passengers['Passengers'], alpha=0.1, color='#1e5fa8')
axes[0].set_title('International Air Passengers (1949–1960)')
axes[0].set_ylabel('Passengers (thousands)')

# Log transform to stabilize variance
log_pass = np.log(passengers['Passengers'])
axes[1].plot(passengers.index, log_pass, color='#e85d20', lw=1.5)
axes[1].set_title('Log(Passengers) — variance stabilized')
axes[1].set_ylabel('log(Passengers)')

plt.tight_layout()
plt.show()
print("📌 Clear upward trend + increasing seasonal swings → multiplicative seasonality")
print("   Log transform converts multiplicative → additive seasonality (constant swing size)")

In [ ]:
# Seasonal plot — one line per year
fig, ax = plt.subplots(figsize=(11, 5))
years = passengers.index.year.unique()
colors_y = plt.cm.viridis(np.linspace(0, 1, len(years)))
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

for year, color in zip(years, colors_y):
    year_data = passengers[passengers.index.year == year]['Passengers'].values
    if len(year_data) == 12:
        ax.plot(range(12), year_data, color=color, lw=1.5, marker='o', markersize=4, label=str(year))

ax.set_xticks(range(12))
ax.set_xticklabels(months)
ax.set_ylabel('Passengers (thousands)')
ax.set_title('Seasonal Plot — Air Passengers by Month\n(one line per year, darker = later)')
ax.legend(title='Year', ncol=4, fontsize=8, loc='upper left')
plt.tight_layout()
plt.show()
print("\n📌 July-August peak every year — summer travel season")
print("   Each year's line is higher → upward trend")
print("   The gap between lines grows → multiplicative seasonality")

## 🔀 Part 2 — Decomposition: Separating the Components

**Additive:** Y_t = Trend_t + Seasonal_t + Remainder_t (use when seasonal amplitude is constant)

**Multiplicative:** Y_t = Trend_t × Seasonal_t × Remainder_t (use when amplitude grows with trend)

**STL (Seasonal and Trend decomposition using Loess)** is more robust than classical decomposition — handles outliers better and allows the seasonal component to change over time.

In [ ]:
# Classical decomposition
decomp = seasonal_decompose(passengers['Passengers'], model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(12, 10))
titles = ['Observed', 'Trend', 'Seasonal', 'Residual']
components = [passengers['Passengers'], decomp.trend, decomp.seasonal, decomp.resid]
colors_d = ['#1e5fa8','#e85d20','#1a7a45','#6b46c1']

for ax, title, comp, color in zip(axes, titles, components, colors_d):
    ax.plot(passengers.index, comp, color=color, lw=1.5)
    ax.set_title(title)
    ax.set_ylabel(title)

plt.suptitle('Classical Multiplicative Decomposition — Air Passengers', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 Seasonal component: a consistent multiplier (1.3 in summer = 30% above trend)")
print("   Trend: smooth upward curve")
print("   Residual: what's left after removing trend and seasonality — should look random")

In [ ]:
# STL Decomposition — more robust
stl = STL(np.log(passengers['Passengers']), period=12, robust=True)
result = stl.fit()

fig = result.plot()
fig.set_size_inches(12, 9)
fig.suptitle('STL Decomposition — log(Air Passengers)\nRobust to outliers, seasonal component can evolve', y=1.01)
plt.tight_layout()
plt.show()

# Strength measures
var_remainder = np.var(result.resid)
var_seasonal_remainder = np.var(result.seasonal + result.resid)
var_trend_remainder    = np.var(result.trend + result.resid)

F_seasonal = max(0, 1 - var_remainder/var_seasonal_remainder)
F_trend    = max(0, 1 - var_remainder/var_trend_remainder)
print(f"\nStrength of trend:     {F_trend:.3f}  (0=none, 1=strong)")
print(f"Strength of seasonality: {F_seasonal:.3f}  (0=none, 1=strong)")
print("\n📌 Both are near 1.0 → very strong trend and seasonality in this series")

## 📉 Part 3 — Autocorrelation: Memory in Time Series

Autocorrelation measures correlation between a series and its *lagged* values:
```
ACF(k) = Corr(y_t, y_{t-k})
```

- Slow decay in ACF → strong trend (non-stationary)
- Spikes at seasonal lags (12, 24, ...) → seasonality
- Quickly decays to near zero → stationary

**PACF** (Partial ACF) removes the influence of intermediate lags — helps identify AR order in ARIMA.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# ACF/PACF for raw series (non-stationary)
plot_acf(passengers['Passengers'], lags=36, ax=axes[0,0], color='#1e5fa8')
axes[0,0].set_title('ACF — Raw Passengers\n(slow decay = trend present = non-stationary)')

plot_pacf(passengers['Passengers'], lags=36, ax=axes[0,1], color='#1e5fa8', method='ywm')
axes[0,1].set_title('PACF — Raw Passengers')

# After differencing (remove trend)
diff_pass = np.log(passengers['Passengers']).diff(12).dropna()  # seasonal differencing
plot_acf(diff_pass, lags=36, ax=axes[1,0], color='#1a7a45')
axes[1,0].set_title('ACF — After Seasonal Differencing\n(much more stationary)')

plot_pacf(diff_pass, lags=36, ax=axes[1,1], color='#1a7a45', method='ywm')
axes[1,1].set_title('PACF — After Seasonal Differencing')

plt.suptitle('ACF and PACF: Raw vs Differenced Series', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 Blue shaded region = 95% confidence band for white noise")
print("   Spikes outside the band are statistically significant autocorrelations")

In [ ]:
# Stationarity test — Augmented Dickey-Fuller
print("=== Augmented Dickey-Fuller Test ===")
print("H₀: Series has a unit root (non-stationary)")
print("Reject H₀ if p-value < 0.05\n")

series_dict = {
    'Raw passengers':           passengers['Passengers'],
    'Log passengers':           np.log(passengers['Passengers']),
    'First difference (log)':   np.log(passengers['Passengers']).diff().dropna(),
    'Seasonal diff (log)':      np.log(passengers['Passengers']).diff(12).dropna(),
}

for name, series in series_dict.items():
    adf_result = adfuller(series)
    stationary = adf_result[1] < 0.05
    print(f"{name:<30}  ADF={adf_result[0]:>7.3f}  p={adf_result[1]:.4f}  {'✓ Stationary' if stationary else '✗ Non-stationary'}")
print("\n📌 Seasonal differencing makes the log series stationary")
print("   Stationary series required for ARIMA modeling (next notebook)")

In [ ]:
# @title 📝 Quiz — Time Series Foundations
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** Name the four components of a time series
# @markdown **Q2:** When would you use multiplicative vs additive decomposition?
# @markdown **Q3:** What does a slowly decaying ACF indicate?
# @markdown **Q4:** What is the difference between seasonality and a cycle?
# @markdown **Q5:** What does 'stationary' mean and why does ARIMA require it?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "Time Series Foundations"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
